# Kouman AI 1.9 — Notebook SFT 2 (V4.3 Multi-Tour & Dioula Riche) sur AMD ROCm

Ce notebook entraîne **Kouman AI 1.9** à partir de la base CPT (**`issoufzousko07/kouman-gemma2-2b-cpt-merged`**) et du dataset SFT V4.3 (**`issoufzousko07/kouman-ia-1.9-sft-v4-3`**), puis fusionne et publie en privé sur **`Tree-AI-lab/kouman-IA-1.9-1.5B-instruct`** avec l'architecture native `model_type: "kouman"`.

### Réglages éprouvés (préservation 100 % CPT + généralisation maximale)
- **Dégel sélectif des 2 seuls tokens `[106, 107]`** (`<start_of_turn>`, `<end_of_turn>`) : les 255 998 autres embeddings CPT restent strictement identiques au bit près.
- **LoRA doux (`r=16, alpha=32, dropout=0.08`)**, **`lr = 4.0e-5`**, **`3.0` époques max**, **`LONGUEUR_MAX_SEQ = 2048`** (pour les dialogues 4–6 tours), masquage `-100` strict sur les tours `user`.
- **Suivi `val_loss` par catégorie & Early Stopping** + **tests OOD ciblés V1.9** (Nicanor anti-OpenAI, bascule FR→DYU, cours Dioula multi-tour, 5 tours relances).

## 1. Environnement AMD ROCm (NumPy >= 2.0 & Transformers 4.51.3)

In [ ]:
import os

os.environ["PYTORCH_HIP_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MIOPEN_LOG_LEVEL"] = "3"
os.environ["MIOPEN_ENABLE_LOGGING"] = "0"
os.environ["MIOPEN_ENABLE_LOGGING_CMD"] = "0"

In [ ]:
%pip install --quiet --upgrade pip
%pip install --quiet --upgrade --upgrade-strategy only-if-needed "numpy>=2.0.0,<2.8.0" "pyarrow>=17.0.0" "transformers>=4.51.0,<5.0.0" "peft>=0.18.0" "accelerate>=0.34.0" "huggingface_hub>=0.27.0,<1.0.0" "safetensors" "tqdm"
%pip install --quiet --force-reinstall --no-deps "transformers==4.51.3" "huggingface_hub==0.30.2" "tokenizers==0.21.4"
%pip install --quiet "regex" "safetensors" "pyyaml" "filelock"

In [ ]:
import numpy as np

if int(np.__version__.split(".")[0]) < 2:
    raise RuntimeError(
        f"[ERREUR] NumPy {np.__version__} en mémoire : exécutez la cellule %pip install puis redémarrez le kernel (Kernel -> Restart)."
    )

import getpass
import json
import math
import random
import shutil
import time
from pathlib import Path

import peft
import pyarrow.parquet as pq
import torch
import transformers
from huggingface_hub import HfApi, hf_hub_download
from peft import LoraConfig, TaskType, get_peft_model
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

if int(transformers.__version__.split(".")[0]) >= 5:
    raise RuntimeError(
        f"[ERREUR] transformers {transformers.__version__} détecté : redémarrez le kernel après %pip install (<5.0.0)."
    )
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("[ERREUR] GPU AMD ROCm / CUDA avec support bfloat16 requis.")

vram_libre_go, vram_totale_go = (x / (1024 ** 3) for x in torch.cuda.mem_get_info(0))
print(f"[OK] GPU actif : {torch.cuda.get_device_name(0)} | VRAM libre : {vram_libre_go:.1f} / {vram_totale_go:.1f} Go")
print(f"transformers={transformers.__version__} | peft={peft.__version__}")

## 2. Configuration Kouman AI 1.9 (SFT V4.3) & Authentification HF

In [ ]:
JETON_HF = os.getenv("HF_TOKEN", "").strip()
if not JETON_HF:
    JETON_HF = getpass.getpass("Jeton Hugging Face (lecture/écriture issoufzousko07 + Tree-AI-lab) : ").strip()
if not JETON_HF:
    raise PermissionError("[ERREUR] Jeton HF obligatoire.")
os.environ["HF_TOKEN"] = JETON_HF

DEPOT_CPT_FUSIONNE = "issoufzousko07/kouman-gemma2-2b-cpt-merged"
DEPOT_DATASET_SFT = "issoufzousko07/kouman-ia-1.9-sft-v4-3"
DEPOT_MODELE_FINAL_SFT = "Tree-AI-lab/kouman-IA-1.9-1.5B-instruct"
PRIVE = True  # Sécurité : publié en privé jusqu'à validation complète (n'écrase pas la V1.8 publique)

# Hyperparamètres SFT V4.3 (calibrés pour 4 800 exemples LIMA max, dialogues jusqu'à 2048 tokens)
LONGUEUR_MAX_SEQ = 2048
TAILLE_MICRO_LOT = 4
PAS_ACCUMULATION = 4              # Lot effectif = 16 exemples / step (~285 steps/époque)
NB_EPOQUES_MAX = 3.0
TAUX_APPRENTISSAGE = 4.0e-5
TAUX_WARMUP = 0.08
WEIGHT_DECAY = 0.05
CLIPPING_NORME = 1.0
INTERVALLE_EVAL_STEPS = 25
PATIENCE_EARLY_STOP = 4
NEFTUNE_ALPHA = 2.0
GRAINE = 42

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.08
MODULES_LORA = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

DOSSIER_SFT = Path("travail_sft_v4_3_v19")
DOSSIER_SFT.mkdir(parents=True, exist_ok=True)
DOSSIER_MEILLEUR_ADAPTATEUR = DOSSIER_SFT / "meilleur_adaptateur_sft"
DOSSIER_MODELE_FUSIONNE = DOSSIER_SFT / "kouman_ia_1_9_instruct_bf16"

client_hf = HfApi(token=JETON_HF)
print(f"Base CPT : {DEPOT_CPT_FUSIONNE}")
print(f"Dataset  : {DEPOT_DATASET_SFT}")
print(f"Sortie   : {DEPOT_MODELE_FINAL_SFT} (privé={PRIVE})")

## 3. Chargement du Dataset SFT V4.3 & Masquage Exact (`labels = -100`)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(DEPOT_CPT_FUSIONNE, token=JETON_HF)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

TEMPLATE_CHAT_KOUMAN = (
    "{{ bos_token }}"
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}<start_of_turn>user\n{{ message['content'] | trim }}<end_of_turn>\n"
    "{% elif message['role'] == 'assistant' or message['role'] == 'model' %}<start_of_turn>model\n{{ message['content'] | trim }}<end_of_turn>\n"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}<start_of_turn>model\n{% endif %}"
)
tokenizer.chat_template = TEMPLATE_CHAT_KOUMAN

ID_BOS = tokenizer.bos_token_id
ID_EOS = tokenizer.eos_token_id
ID_START_OF_TURN = tokenizer.convert_tokens_to_ids("<start_of_turn>")
ID_END_OF_TURN = tokenizer.convert_tokens_to_ids("<end_of_turn>")
if ID_BOS is None or ID_EOS is None or ID_START_OF_TURN in (None, tokenizer.unk_token_id) or ID_END_OF_TURN in (None, tokenizer.unk_token_id):
    raise ValueError(f"[ERREUR] Tokens spéciaux invalides : bos={ID_BOS}, eos={ID_EOS}, sot={ID_START_OF_TURN}, eot={ID_END_OF_TURN}")

IDS_ARRET = [ID_EOS, ID_END_OF_TURN]
print(f"[OK] Tokens spéciaux vérifiés : BOS={ID_BOS}, EOS={ID_EOS}, <start_of_turn>={ID_START_OF_TURN}, <end_of_turn>={ID_END_OF_TURN}")


def charger_parquets_sft(nom_split: str) -> list[dict]:
    chemin = hf_hub_download(
        repo_id=DEPOT_DATASET_SFT,
        filename=f"data/{nom_split}-00000-of-00001.parquet",
        repo_type="dataset",
        token=JETON_HF,
        force_download=True,
    )
    table = pq.read_table(chemin, columns=["messages", "categorie"])
    return table.to_pylist()


class DatasetChatKoumanSFT(Dataset):
    """Encode les dialogues multi-tours avec IDs exacts <start_of_turn>/<end_of_turn> et masquage (-100) sur user."""

    def __init__(self, lignes: list[dict], max_len: int):
        self.exemples: list[tuple[torch.Tensor, torch.Tensor, str]] = []
        ignores = 0

        tok_user_nl = tokenizer.encode("user\n", add_special_tokens=False)
        tok_nl = tokenizer.encode("\n", add_special_tokens=False)
        tok_model_nl = tokenizer.encode("model\n", add_special_tokens=False)

        for item in lignes:
            msgs = item["messages"]
            if len(msgs) < 2 or len(msgs) % 2 != 0:
                raise ValueError(f"[ERREUR] Format messages inattendu (nombre impair/vide) : {msgs}")

            ids_complets: list[int] = [ID_BOS]
            labels_complets: list[int] = [-100]
            nb_tours = len(msgs) // 2

            for idx_tour in range(nb_tours):
                m_user = msgs[2 * idx_tour]
                m_asst = msgs[2 * idx_tour + 1]
                if m_user["role"] != "user" or m_asst["role"] != "assistant":
                    raise ValueError(f"[ERREUR] Alternance user/assistant invalide : {msgs}")

                bloc_user = (
                    [ID_START_OF_TURN]
                    + tok_user_nl
                    + tokenizer.encode(m_user["content"].strip(), add_special_tokens=False)
                    + [ID_END_OF_TURN]
                    + tok_nl
                    + [ID_START_OF_TURN]
                    + tok_model_nl
                )
                bloc_asst = tokenizer.encode(m_asst["content"].strip(), add_special_tokens=False) + [ID_END_OF_TURN]

                ids_complets.extend(bloc_user)
                labels_complets.extend([-100] * len(bloc_user))
                ids_complets.extend(bloc_asst)
                labels_complets.extend(bloc_asst)

                if idx_tour < nb_tours - 1:
                    ids_complets.extend(tok_nl)
                    labels_complets.extend([-100] * len(tok_nl))

            ids_complets = ids_complets[:max_len]
            labels_complets = labels_complets[:max_len]
            if sum(1 for x in labels_complets if x != -100) < 3:
                ignores += 1
                continue

            self.exemples.append(
                (
                    torch.tensor(ids_complets, dtype=torch.long),
                    torch.tensor(labels_complets, dtype=torch.long),
                    item.get("categorie", "inconnue"),
                )
            )

        if not self.exemples:
            raise RuntimeError("[ERREUR] Aucun exemple SFT valide après encodage.")
        print(f"   • {len(self.exemples):,} exemples encodés ({ignores} trop longs ignorés)")

    def __len__(self) -> int:
        return len(self.exemples)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        return self.exemples[idx]


def assembler_lot_padding(batch: list[tuple[torch.Tensor, torch.Tensor, str]]) -> dict:
    seqs_ids, seqs_labels, cats = zip(*batch)
    input_ids = pad_sequence(seqs_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    labels = pad_sequence(seqs_labels, batch_first=True, padding_value=-100)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels, "categories": list(cats)}


print("Téléchargement et tokenisation du dataset SFT V4.3...")
dataset_train = DatasetChatKoumanSFT(charger_parquets_sft("train"), LONGUEUR_MAX_SEQ)
dataset_val = DatasetChatKoumanSFT(charger_parquets_sft("validation"), LONGUEUR_MAX_SEQ)

g = torch.Generator().manual_seed(GRAINE)
chargeur_train = DataLoader(
    dataset_train, batch_size=TAILLE_MICRO_LOT, shuffle=True, generator=g,
    collate_fn=assembler_lot_padding, pin_memory=True,
)
chargeur_val = DataLoader(
    dataset_val, batch_size=TAILLE_MICRO_LOT, shuffle=False,
    collate_fn=assembler_lot_padding, pin_memory=True,
)

## 3B. Audit Visuel des Extraits du Jeu d'Entraînement (Masquage `-100` avant GPU)

In [ ]:
import random as _rnd
from collections import defaultdict as _dd

par_cat_visu = _dd(list)
for idx_v, (ids_t, lab_t, cat_t) in enumerate(dataset_train.exemples):
    par_cat_visu[cat_t].append((idx_v, ids_t, lab_t))

rng_v = _rnd.Random(GRAINE)
print("=" * 78)
print(f"AUDIT VISUEL DU DATASET V4.3 CHARGÉ ({len(dataset_train):,} train | {len(dataset_val):,} val)")
print("=" * 78)
for cat_v in sorted(par_cat_visu):
    liste_v = par_cat_visu[cat_v]
    print(f"\nCatégorie : {cat_v} ({len(liste_v):,} exemples en train)")
    for idx_v, ids_t, lab_t in rng_v.sample(liste_v, min(2, len(liste_v))):
        nb_masques = int((lab_t == -100).sum().item())
        nb_sup = int((lab_t != -100).sum().item())
        ids_sup = [int(x) for x in lab_t.tolist() if x != -100]
        texte_complet = tokenizer.decode(ids_t.tolist(), skip_special_tokens=False)
        texte_sup = tokenizer.decode(ids_sup, skip_special_tokens=False)
        print(f"  • Exemple #{idx_v} | total={len(ids_t)} tok | masqués(-100)={nb_masques} | supervisés={nb_sup}")
        print(f"    [PROMPT+REP] : {texte_complet[:420]}{'...' if len(texte_complet) > 420 else ''}")
        print(f"    [SUPERVISÉ]  : {texte_sup[:320]}{'...' if len(texte_sup) > 320 else ''}")

## 4. Chargement du Modèle CPT & LoRA SFT + Dégel Sélectif `[106, 107]`

In [ ]:
print(f"Chargement de {DEPOT_CPT_FUSIONNE} en bfloat16 (attn=eager)...")
modele_base = AutoModelForCausalLM.from_pretrained(
    DEPOT_CPT_FUSIONNE,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    token=JETON_HF,
)
modele_base.config.use_cache = False

vram_libre_go = torch.cuda.mem_get_info(0)[0] / (1024 ** 3)
if vram_libre_go < 100.0:
    print(f"VRAM libre {vram_libre_go:.1f} Go (< 100 Go) : activation Gradient Checkpointing.")
    modele_base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    modele_base.enable_input_require_grads()

config_lora_sft = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    use_rslora=False,
    bias="none",
    target_modules=MODULES_LORA,
    task_type=TaskType.CAUSAL_LM,
)
modele = get_peft_model(modele_base, config_lora_sft).to(device="cuda", dtype=torch.bfloat16)

# Dégel SÉLECTIF des 2 seuls tokens <start_of_turn> (106) et <end_of_turn> (107)
IDS_SPECIAUX_SFT = [ID_START_OF_TURN, ID_END_OF_TURN]
poids_embed = modele.get_input_embeddings().weight
poids_embed.requires_grad_(True)
ID_TEMOIN_CPT = 5000
temoin_emb_avant = poids_embed[ID_TEMOIN_CPT].detach().clone().cpu()
masque_grad_speciaux = torch.zeros(poids_embed.size(0), 1, device=poids_embed.device, dtype=poids_embed.dtype)
masque_grad_speciaux[IDS_SPECIAUX_SFT] = 1.0
handle_masque_emb = poids_embed.register_hook(lambda grad: grad * masque_grad_speciaux)
print(f"Dégel sélectif actif sur 2 tokens spéciaux uniquement : {IDS_SPECIAUX_SFT} (255 998 autres tokens strictement gelés)")


def hook_neftune(module, _entrees, sortie):
    if module.training and NEFTUNE_ALPHA > 0:
        dims = torch.tensor(sortie.size(1) * sortie.size(2), device=sortie.device, dtype=sortie.dtype)
        mag_norm = NEFTUNE_ALPHA / torch.sqrt(dims)
        return sortie + torch.empty_like(sortie).uniform_(-1.0, 1.0) * mag_norm
    return sortie


handle_neftune = modele.get_input_embeddings().register_forward_hook(hook_neftune)

tot = sum(p.numel() for p in modele.parameters())
entr = sum(p.numel() for p in modele.parameters() if p.requires_grad)
print(f"Paramètres SFT entraînables : {entr:,} / {tot:,} ({100 * entr / tot:.2f} % — NEFTune={NEFTUNE_ALPHA})")

## 5. Boucle SFT V4.3 avec `val_loss` par Catégorie & Arrêt Anticipé

In [ ]:
from collections import defaultdict
import torch.nn.functional as F

steps_par_epoque = math.ceil(len(chargeur_train) / PAS_ACCUMULATION)
total_steps_max = int(round(steps_par_epoque * NB_EPOQUES_MAX))
steps_warmup = max(10, int(total_steps_max * TAUX_WARMUP))

params_lora = [p for n, p in modele.named_parameters() if p.requires_grad and p is not poids_embed]
optimiseur = torch.optim.AdamW(
    [
        {"params": params_lora, "lr": TAUX_APPRENTISSAGE, "lr_base": TAUX_APPRENTISSAGE, "weight_decay": WEIGHT_DECAY},
        {"params": [poids_embed], "lr": TAUX_APPRENTISSAGE * 3.0, "lr_base": TAUX_APPRENTISSAGE * 3.0, "weight_decay": 0.0},
    ],
    betas=(0.9, 0.99),
)


def sauvegarder_meilleur_sft(dossier_cible: Path) -> None:
    modele.save_pretrained(str(dossier_cible))
    torch.save(
        {
            "ids": IDS_SPECIAUX_SFT,
            "weights": modele.get_input_embeddings().weight[IDS_SPECIAUX_SFT].detach().clone().cpu(),
        },
        dossier_cible / "tokens_speciaux_sft.pt",
    )


def facteur_lr_cosine(step_actuel: int) -> float:
    if step_actuel < steps_warmup:
        return float(step_actuel + 1) / float(steps_warmup)
    progression = min(1.0, (step_actuel - steps_warmup) / float(max(1, total_steps_max - steps_warmup)))
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progression))


@torch.no_grad()
def evaluer_val_loss() -> tuple[float, dict[str, float]]:
    modele.eval()
    somme_tot = 0.0
    nb_tok_tot = 0
    somme_cat: dict[str, float] = defaultdict(float)
    tok_cat: dict[str, int] = defaultdict(int)
    for lot in chargeur_val:
        cats = lot["categories"]
        lot_cuda = {k: v.to("cuda", non_blocking=True) for k, v in lot.items() if k != "categories"}
        with torch.amp.autocast("cuda", dtype=torch.bfloat16):
            logits = modele(input_ids=lot_cuda["input_ids"], attention_mask=lot_cuda["attention_mask"]).logits
        shift_logits = logits[:, :-1, :].float()
        shift_labels = lot_cuda["labels"][:, 1:]
        pertes_tok = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=-100,
            reduction="none",
        ).view(shift_labels.shape)
        masque = shift_labels != -100
        for i, cat in enumerate(cats):
            n_t = int(masque[i].sum().item())
            if n_t > 0:
                s_i = float(pertes_tok[i][masque[i]].sum().item())
                somme_tot += s_i
                nb_tok_tot += n_t
                somme_cat[cat] += s_i
                tok_cat[cat] += n_t
    torch.cuda.empty_cache()
    modele.train()
    val_globale = somme_tot / max(1, nb_tok_tot)
    par_cat = {c: round(somme_cat[c] / max(1, tok_cat[c]), 4) for c in sorted(somme_cat)}
    return val_globale, par_cat


val_loss_init, val_cats_init = evaluer_val_loss()
print(f"Validation initiale avant SFT : val_loss={val_loss_init:.4f} | par catégorie={val_cats_init}")

meilleure_val_loss = val_loss_init
sans_amelioration = 0
step_global = 0
arret_anticipe = False
historique_sft = []
fenetre_train: list[float] = []

modele.train()
optimiseur.zero_grad(set_to_none=True)
compteur_accum = 0
perte_accum = 0.0

barre = tqdm(total=total_steps_max, desc="SFT V4.3 (Kouman 1.9)")
iterateur = iter(chargeur_train)

while step_global < total_steps_max and not arret_anticipe:
    try:
        lot = next(iterateur)
    except StopIteration:
        iterateur = iter(chargeur_train)
        lot = next(iterateur)

    lot_cuda = {k: v.to("cuda", non_blocking=True) for k, v in lot.items() if k != "categories"}
    with torch.amp.autocast("cuda", dtype=torch.bfloat16):
        perte = modele(**lot_cuda).loss / PAS_ACCUMULATION

    perte.backward()
    perte_accum += float(perte.detach().cpu()) * PAS_ACCUMULATION
    compteur_accum += 1

    if compteur_accum >= PAS_ACCUMULATION:
        mult = facteur_lr_cosine(step_global)
        for grp in optimiseur.param_groups:
            grp["lr"] = grp["lr_base"] * mult

        torch.nn.utils.clip_grad_norm_([p for p in modele.parameters() if p.requires_grad], CLIPPING_NORME)
        optimiseur.step()
        optimiseur.zero_grad(set_to_none=True)

        train_loss_step = perte_accum / compteur_accum
        fenetre_train.append(train_loss_step)
        if len(fenetre_train) > INTERVALLE_EVAL_STEPS:
            fenetre_train.pop(0)
        train_moy = sum(fenetre_train) / len(fenetre_train)
        perte_accum = 0.0
        compteur_accum = 0
        step_global += 1
        barre.update(1)
        barre.set_postfix(train_moy=f"{train_moy:.3f}", best_val=f"{meilleure_val_loss:.3f}")

        if step_global % INTERVALLE_EVAL_STEPS == 0 or step_global == total_steps_max:
            val_loss, val_cats = evaluer_val_loss()
            historique_sft.append({
                "step": step_global,
                "train_moy": round(train_moy, 4),
                "val_loss": round(val_loss, 4),
                "val_categories": val_cats,
            })
            if val_loss < meilleure_val_loss - 0.002:
                meilleure_val_loss = val_loss
                sans_amelioration = 0
                sauvegarder_meilleur_sft(DOSSIER_MEILLEUR_ADAPTATEUR)
                print(f"\nStep {step_global} | train_moy={train_moy:.4f} | val={val_loss:.4f} (nouveau meilleur [OK]) | {val_cats}")
            else:
                sans_amelioration += 1
                print(f"\n[PAUSE] Step {step_global} | train_moy={train_moy:.4f} | val={val_loss:.4f} (stagnation {sans_amelioration}/{PATIENCE_EARLY_STOP}) | {val_cats}")
                if sans_amelioration >= PATIENCE_EARLY_STOP:
                    print("[ARRET] Arrêt anticipé activé pour préserver la généralisation.")
                    arret_anticipe = True

barre.close()
handle_neftune.remove()
handle_masque_emb.remove()
assert torch.equal(modele.get_input_embeddings().weight[ID_TEMOIN_CPT].detach().cpu(), temoin_emb_avant), "[ERREUR] Erreur : un embedding CPT non-spécial a dérivé !"
print("[OK] Contrôle bit-à-bit OK : les 255 998 embeddings CPT sont restés strictement identiques.")
if not DOSSIER_MEILLEUR_ADAPTATEUR.exists():
    sauvegarder_meilleur_sft(DOSSIER_MEILLEUR_ADAPTATEUR)
print(f"SFT V4.3 terminé | meilleur val_loss={meilleure_val_loss:.4f} | adaptateur={DOSSIER_MEILLEUR_ADAPTATEUR}")

## 6. Tests OOD Ciblés V1.9 (Nicanor Anti-OpenAI, Bascule FR→DYU, Cours Dioula Multi-Tour, 5 Tours Relances)

In [ ]:
from peft import PeftModel

del modele, modele_base
torch.cuda.empty_cache()
modele_base_propre = AutoModelForCausalLM.from_pretrained(
    DEPOT_CPT_FUSIONNE,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    token=JETON_HF,
)
chemin_spec = DOSSIER_MEILLEUR_ADAPTATEUR / "tokens_speciaux_sft.pt"
if not chemin_spec.exists():
    raise FileNotFoundError(f"[ERREUR] Vecteurs tokens spéciaux introuvables : {chemin_spec}")
etat_spec = torch.load(chemin_spec, map_location="cpu", weights_only=True)
with torch.no_grad():
    emb_in = modele_base_propre.get_input_embeddings().weight
    emb_in[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_in.dtype)
    emb_out = modele_base_propre.get_output_embeddings().weight
    if emb_out.data_ptr() != emb_in.data_ptr():
        emb_out[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_out.dtype)
print(f"[OK] Vecteurs entraînés <start_of_turn>/<end_of_turn> réinjectés : ids={etat_spec['ids']}")

modele_eval = PeftModel.from_pretrained(
    modele_base_propre,
    str(DOSSIER_MEILLEUR_ADAPTATEUR),
    is_trainable=False,
).to(device="cuda", dtype=torch.bfloat16)
modele_eval.eval()

QUESTIONS_OOD_V19 = [
    ("Identité croisée OOD (Nicanor)", "bonjour je suis nicanor et toi ?"),
    ("Bascule langue OOD (FR -> DYU)", "bonjour repond moi en dioula"),
    ("Double langue OOD (FR + BCI)", "presente toi en français puis en baoulé"),
    ("Dioula encyclopédique OOD", "Hakili min be sɔrɔ mɔgɔ bolo, o ye mun lo ye?"),
    ("Dioula cours direct OOD", "Jatikalan kalan n ye."),
    ("Calcul 4 chiffres (OOD)", "combien font 2345+5688"),
    ("Traduction FR -> BCI (OOD)", "traduis en baoulé : demain matin je vais saluer mes grands-parents au village"),
    ("Traduction FR -> DYU (OOD)", "traduis en dioula : ma sœur prépare du riz chaud pour le dîner de la famille"),
    ("Traduction EN -> DYU (OOD)", "Translate into Dioula: Good health is more important than money."),
    ("Traduction EN -> BCI (OOD)", "Translate into Baoulé: Clean rivers help every village live better."),
]

for titre, question in QUESTIONS_OOD_V19:
    prompt_chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        tokenize=False,
        add_generation_prompt=True,
    )
    entrees = tokenizer(prompt_chat, return_tensors="pt", add_special_tokens=False).to("cuda")
    torch.manual_seed(GRAINE)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        sortie = modele_eval.generate(
            **entrees,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.08,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=IDS_ARRET,
        )
    gen_ids = sortie[0, entrees["input_ids"].shape[1]:].tolist()
    arret = bool(gen_ids) and gen_ids[-1] in IDS_ARRET
    rep = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    print("=" * 72)
    print(f"{titre} | {'[OK] EOS' if arret else '[ERREUR] MAX'} ({len(gen_ids)} tok)\nUtilisateur : {question}\nAssistant : {rep}")


def executer_test_multitour(nom_scenario: str, tours_user: list[str], max_tours_memoire: int = 4):
    print("=" * 72)
    print(f"MULTI-TOUR OOD — {nom_scenario}")
    hist: list[dict[str, str]] = []
    for idx_u, msg_u in enumerate(tours_user, start=1):
        hist.append({"role": "user", "content": msg_u})
        fenetre = hist[-(max_tours_memoire * 2 + 1):]
        if fenetre[0]["role"] != "user":
            fenetre = fenetre[1:]
        p_mt = tokenizer.apply_chat_template(fenetre, tokenize=False, add_generation_prompt=True)
        entrees_mt = tokenizer(p_mt, return_tensors="pt", add_special_tokens=False).to("cuda")
        torch.manual_seed(GRAINE + idx_u)
        with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
            sortie_mt = modele_eval.generate(
                **entrees_mt,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.45,
                top_p=0.9,
                repetition_penalty=1.08,
                use_cache=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=IDS_ARRET,
            )
        rep_mt = tokenizer.decode(sortie_mt[0, entrees_mt["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        hist.append({"role": "assistant", "content": rep_mt})
        print(f"  [Tour {idx_u}] Utilisateur : {msg_u}\n  [Tour {idx_u}] Assistant : {rep_mt}")


# Scénario A : Dioula conversationnel + cours + changement de sujet au tour 4 (vérifie zéro timidité + zéro perte au tour 4)
executer_test_multitour(
    "Dioula Cours + Changement Sujet (4 tours)",
    [
        "I ni sɔgɔma",
        "Nne ka kɛnɛ, n b'a fɛ ka jatebɔ dege",
        "Jatikalan kalan n ye.",
        "Hakili min be sɔrɔ mɔgɔ bolo, o ye mun lo ye?",
    ],
)

# Scénario B : 5 tours relances courtes + bascule baoulé + identité croisée (vérifie zéro 'Au revoir' ni OpenAI)
executer_test_multitour(
    "Relances Courtes + Bascule + Identité (5 tours)",
    [
        "Comment on dit bonjour et bonsoir en baoulé et en dioula ?",
        "Comment on dit merci beaucoup en anglais et en dioula ?",
        "et comment on dit je rentre à la maison maintenant en dioula ?",
        "traduis en baoulé : la vie est belle au village",
        "bonjour je suis nicanor et toi ?",
    ],
)

## 7. Fusion Finale, Architecture Native `model_type="kouman"` & Publication Privée (`Tree-AI-lab/kouman-IA-1.9-1.5B-instruct`)

In [ ]:
# Sécurité : passez à True UNIQUEMENT après avoir validé les tests OOD de la cellule 6 !
PUBLIER_MODELE_FINAL = False

CODE_CONFIG_KOUMAN = '''"""Configuration officielle du modèle Kouman AI (Tree AI Lab)."""

from transformers.models.gemma2.configuration_gemma2 import Gemma2Config


class KoumanConfig(Gemma2Config):
    model_type = "kouman"
'''

CODE_MODELING_KOUMAN = '''"""Implémentation officielle du modèle Kouman AI (Tree AI Lab)."""

from transformers.models.gemma2.modeling_gemma2 import Gemma2ForCausalLM, Gemma2Model

from .configuration_kouman import KoumanConfig


class KoumanModel(Gemma2Model):
    config_class = KoumanConfig


class KoumanForCausalLM(Gemma2ForCausalLM):
    config_class = KoumanConfig
'''

if not PUBLIER_MODELE_FINAL:
    print("[PAUSE] Publication désactivée par sécurité (PUBLIER_MODELE_FINAL = False).")
    print("Vérifiez les tests mono-tour et multi-tour ci-dessus, puis passez PUBLIER_MODELE_FINAL = True pour fusionner et publier.")
else:
    if "modele_eval" not in globals():
        from peft import PeftModel
        print("Rechargement propre du meilleur adaptateur + tokens spéciaux avant fusion...")
        modele_base_propre = AutoModelForCausalLM.from_pretrained(
            DEPOT_CPT_FUSIONNE, torch_dtype=torch.bfloat16, attn_implementation="eager", token=JETON_HF
        )
        etat_spec = torch.load(DOSSIER_MEILLEUR_ADAPTATEUR / "tokens_speciaux_sft.pt", map_location="cpu", weights_only=True)
        with torch.no_grad():
            emb_in = modele_base_propre.get_input_embeddings().weight
            emb_in[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_in.dtype)
            emb_out = modele_base_propre.get_output_embeddings().weight
            if emb_out.data_ptr() != emb_in.data_ptr():
                emb_out[etat_spec["ids"]] = etat_spec["weights"].to(dtype=emb_out.dtype)
        modele_eval = PeftModel.from_pretrained(
            modele_base_propre, str(DOSSIER_MEILLEUR_ADAPTATEUR), is_trainable=False
        ).to(device="cuda", dtype=torch.bfloat16)

    print("Fusion finale du meilleur adaptateur SFT V4.3 sur le modèle CPT...")
    modele_final = modele_eval.merge_and_unload()
    modele_final.generation_config.eos_token_id = IDS_ARRET
    modele_final.generation_config.pad_token_id = tokenizer.pad_token_id
    modele_final.generation_config.do_sample = True
    modele_final.generation_config.temperature = 0.45
    modele_final.generation_config.top_p = 0.9
    modele_final.generation_config.repetition_penalty = 1.08

    DOSSIER_MODELE_FUSIONNE.mkdir(parents=True, exist_ok=True)
    modele_final.save_pretrained(str(DOSSIER_MODELE_FUSIONNE), safe_serialization=True)
    tokenizer.chat_template = TEMPLATE_CHAT_KOUMAN
    tokenizer.save_pretrained(str(DOSSIER_MODELE_FUSIONNE))

    # Enregistrement natif model_type="kouman" + auto_map + use_cache=True (zéro tag gemma2 sur HF)
    chemin_cfg = DOSSIER_MODELE_FUSIONNE / "config.json"
    cfg = json.loads(chemin_cfg.read_text(encoding="utf-8"))
    cfg["model_type"] = "kouman"
    cfg["architectures"] = ["KoumanForCausalLM"]
    cfg["auto_map"] = {
        "AutoConfig": "configuration_kouman.KoumanConfig",
        "AutoModel": "modeling_kouman.KoumanModel",
        "AutoModelForCausalLM": "modeling_kouman.KoumanForCausalLM",
    }
    cfg["use_cache"] = True
    chemin_cfg.write_text(json.dumps(cfg, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    (DOSSIER_MODELE_FUSIONNE / "configuration_kouman.py").write_text(CODE_CONFIG_KOUMAN, encoding="utf-8")
    (DOSSIER_MODELE_FUSIONNE / "modeling_kouman.py").write_text(CODE_MODELING_KOUMAN, encoding="utf-8")

    (DOSSIER_MODELE_FUSIONNE / "historique_sft_v4_3_v19.json").write_text(
        json.dumps(historique_sft, indent=2, ensure_ascii=False), encoding="utf-8"
    )

    code_inference = "\n".join([
        "#!/usr/bin/env python3",
        "# Script d'inférence officiel pour Kouman AI 1.9 (Baoulé, Dioula, Français, Anglais)",
        "import argparse",
        "import torch",
        "from transformers import AutoModelForCausalLM, AutoTokenizer",
        "",
        f"DEPOT_DEFAUT = {DEPOT_MODELE_FINAL_SFT!r}",
        "",
        "",
        "def charger_kouman(depot: str = DEPOT_DEFAUT, jeton_hf: str | None = None):",
        "    tokenizer = AutoTokenizer.from_pretrained(depot, token=jeton_hf)",
        "    if tokenizer.pad_token_id is None:",
        "        tokenizer.pad_token_id = tokenizer.eos_token_id",
        "    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32",
        "    modele = AutoModelForCausalLM.from_pretrained(",
        "        depot,",
        "        torch_dtype=dtype,",
        "        device_map='auto' if torch.cuda.is_available() else None,",
        "        attn_implementation='eager',",
        "        trust_remote_code=True,",
        "        token=jeton_hf,",
        "    )",
        "    modele.eval()",
        "    return modele, tokenizer",
        "",
        "",
        "def repondre_messages(modele, tokenizer, messages: list[dict[str, str]], max_new_tokens: int | None = None, temperature: float = 0.45) -> str:",
        "    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)",
        "    entrees = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(modele.device)",
        "    id_eot = tokenizer.convert_tokens_to_ids('<end_of_turn>')",
        "    ids_arret = [i for i in (tokenizer.eos_token_id, id_eot) if isinstance(i, int) and i >= 0]",
        "    limite_ctx = getattr(modele.config, 'max_position_embeddings', 8192)",
        "    tokens_restants = limite_ctx - entrees['input_ids'].shape[1]",
        "    if tokens_restants <= 0:",
        "        raise ValueError('Fenêtre de contexte maximale atteinte.')",
        "    budget_tokens = tokens_restants if max_new_tokens is None else min(max_new_tokens, tokens_restants)",
        "    with torch.no_grad():",
        "        sortie = modele.generate(",
        "            **entrees,",
        "            max_new_tokens=budget_tokens,",
        "            do_sample=temperature > 0,",
        "            temperature=max(temperature, 1e-4),",
        "            top_p=0.9,",
        "            repetition_penalty=1.08,",
        "            pad_token_id=tokenizer.pad_token_id,",
        "            eos_token_id=ids_arret,",
        "        )",
        "    return tokenizer.decode(sortie[0, entrees['input_ids'].shape[1]:], skip_special_tokens=True).strip()",
        "",
        "",
        "def repondre(modele, tokenizer, question: str, max_new_tokens: int | None = None, temperature: float = 0.45) -> str:",
        "    return repondre_messages(modele, tokenizer, [{'role': 'user', 'content': question}], max_new_tokens, temperature)",
        "",
        "",
        "def main():",
        "    parseur = argparse.ArgumentParser(description='Inférence Kouman AI 1.9 (Tree AI Lab)')",
        "    parseur.add_argument('--modele', default=DEPOT_DEFAUT)",
        "    parseur.add_argument('--question', default=None, help='Question unique ; si absent, lance le mode chat multi-tour.')",
        "    parseur.add_argument('--max-tokens', type=int, default=None, help='Plafond optionnel (par défaut : arrêt naturel sans limite artificielle).')",
        "    parseur.add_argument('--max-tours', type=int, default=4, help='Fenêtre glissante de tours conservés en mémoire.')",
        "    args = parseur.parse_args()",
        "    modele, tokenizer = charger_kouman(args.modele)",
        "    if args.question:",
        "        print(repondre(modele, tokenizer, args.question, max_new_tokens=args.max_tokens))",
        "        return",
        "    print('Kouman AI 1.9 prêt (tapez /reset pour vider la mémoire, quitter pour sortir).')",
        "    historique: list[dict[str, str]] = []",
        "    while True:",
        "        q = input('Utilisateur Vous : ').strip()",
        "        if not q or q.lower() in ('quitter', 'exit', 'quit'):",
        "            break",
        "        if q.lower() == '/reset':",
        "            historique.clear()",
        "            print('Historique effacé.')",
        "            continue",
        "        historique.append({'role': 'user', 'content': q})",
        "        fenetre = historique[-(args.max_tours * 2 + 1):]",
        "        if fenetre[0]['role'] != 'user':",
        "            fenetre = fenetre[1:]",
        "        rep = repondre_messages(modele, tokenizer, fenetre, max_new_tokens=args.max_tokens)",
        "        historique.append({'role': 'assistant', 'content': rep})",
        "        print('Assistant Kouman AI :', rep)",
        "",
        "",
        "if __name__ == '__main__':",
        "    main()",
    ]) + "\n"
    (DOSSIER_MODELE_FUSIONNE / "inference_kouman.py").write_text(code_inference, encoding="utf-8")

    readme_md = "\n".join([
        "---",
        "language:",
        "- bci",
        "- dyu",
        "- fr",
        "- en",
        "pipeline_tag: text-generation",
        "tags:",
        "- kouman-ai",
        "- tree-ai-lab",
        "- baoule",
        "- dioula",
        "- cote-divoire",
        "- african-languages",
        "---",
        "",
        "# Kouman AI 1.9 Instruct (Baoulé · Dioula · Français · Anglais)",
        "",
        "Modèle conversationnel quadrilingue développé par **Tree AI Lab** (Côte d'Ivoire) avec alignement multi-tour renforcé, Dioula pédagogique étendu et identité naturelle.",
        "",
        f"- **Meilleure `val_loss` SFT V4.3** : `{meilleure_val_loss:.4f}`",
        "- **Script autonome inclus** : `inference_kouman.py`",
        "",
        "## Utilisation rapide (`transformers`)",
        "",
        "```python",
        "import torch",
        "from transformers import AutoModelForCausalLM, AutoTokenizer",
        "",
        f"depot = {DEPOT_MODELE_FINAL_SFT!r}",
        "tokenizer = AutoTokenizer.from_pretrained(depot)",
        "modele = AutoModelForCausalLM.from_pretrained(",
        "    depot,",
        "    torch_dtype=torch.bfloat16,",
        "    device_map='auto',",
        "    attn_implementation='eager',",
        "    trust_remote_code=True,",
        ")",
        "",
        "messages = [{'role': 'user', 'content': 'I ni sɔgɔma ! I tɔgɔ ye di ?'}]",
        "prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)",
        "entrees = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(modele.device)",
        "ids_arret = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<end_of_turn>')]",
        "tokens_restants = modele.config.max_position_embeddings - entrees['input_ids'].shape[1]",
        "",
        "sortie = modele.generate(",
        "    **entrees,",
        "    max_new_tokens=tokens_restants,",
        "    do_sample=True,",
        "    temperature=0.45,",
        "    top_p=0.9,",
        "    repetition_penalty=1.08,",
        "    eos_token_id=ids_arret,",
        ")",
        "print(tokenizer.decode(sortie[0, entrees['input_ids'].shape[1]:], skip_special_tokens=True))",
        "```",
    ]) + "\n"
    (DOSSIER_MODELE_FUSIONNE / "README.md").write_text(readme_md, encoding="utf-8")

    client_hf.create_repo(repo_id=DEPOT_MODELE_FINAL_SFT, repo_type="model", private=PRIVE, exist_ok=True)
    client_hf.upload_folder(
        folder_path=str(DOSSIER_MODELE_FUSIONNE),
        repo_id=DEPOT_MODELE_FINAL_SFT,
        repo_type="model",
        commit_message=f"feat: Kouman AI 1.9 Instruct (SFT V4.3 multi-tour + Dioula riche + identité 4.5%, model_type=kouman, val_loss={meilleure_val_loss:.4f})",
    )
    print(f"Kouman AI 1.9 publié : https://huggingface.co/{DEPOT_MODELE_FINAL_SFT} (privé={PRIVE})")

## 8. Vérification Post-Fusion (`model_type="kouman"` rechargé en autonome sans PEFT)

In [ ]:
import gc

for var_nom in ("modele_eval", "modele_final", "modele_base_propre"):
    if var_nom in globals():
        del globals()[var_nom]
gc.collect()
torch.cuda.empty_cache()

if not DOSSIER_MODELE_FUSIONNE.exists():
    raise FileNotFoundError(
        f"[ERREUR] Le modèle fusionné local '{DOSSIER_MODELE_FUSIONNE}' n'existe pas encore. "
        "Passez PUBLIER_MODELE_FINAL = True en Cellule 7 avant d'exécuter la Cellule 8."
    )

source_fusion = str(DOSSIER_MODELE_FUSIONNE)
print(f"Rechargement autonome (sans PEFT, trust_remote_code=True) depuis : {source_fusion}")

tok_fusion = AutoTokenizer.from_pretrained(source_fusion, token=JETON_HF)
mod_fusion = AutoModelForCausalLM.from_pretrained(
    source_fusion,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
    trust_remote_code=True,
    token=JETON_HF,
).to("cuda")
mod_fusion.eval()

assert mod_fusion.config.model_type == "kouman", f"[ERREUR] model_type inattendu : {mod_fusion.config.model_type}"
nb_nan = sum(int(torch.isnan(p).any().item() or torch.isinf(p).any().item()) for p in mod_fusion.parameters())
assert nb_nan == 0, f"[ERREUR] {nb_nan} tenseur(s) contiennent NaN/Inf après fusion !"
assert tok_fusion.chat_template is not None, "[ERREUR] chat_template manquant !"
id_eot_f = tok_fusion.convert_tokens_to_ids("<end_of_turn>")
ids_arret_f = [i for i in (tok_fusion.eos_token_id, id_eot_f) if i is not None]
print(f"[OK] Intégrité OK : model_type='kouman' | 0 NaN/Inf | chat_template OK | eos_token_id={mod_fusion.generation_config.eos_token_id}")

TESTS_POST_MERGE_V19 = [
    ("Identité Nicanor (OOD)", "bonjour je suis nicanor et toi ?"),
    ("Bascule FR->DYU (OOD)", "bonjour repond moi en dioula"),
    ("Double langue FR+BCI (OOD)", "presente toi en français puis en baoulé"),
    ("Dioula cours (OOD)", "Jatikalan kalan n ye."),
    ("Traduction FR->DYU (OOD)", "traduis en dioula : ma sœur prépare du riz chaud pour le dîner de la famille"),
    ("Traduction FR->BCI (OOD)", "traduis en baoulé : demain matin je vais saluer mes grands-parents au village"),
]

ok_arret = 0
for titre_m, q_m in TESTS_POST_MERGE_V19:
    p_m = tok_fusion.apply_chat_template([{"role": "user", "content": q_m}], tokenize=False, add_generation_prompt=True)
    inp_m = tok_fusion(p_m, return_tensors="pt", add_special_tokens=False).to("cuda")
    torch.manual_seed(GRAINE)
    with torch.no_grad():
        out_m = mod_fusion.generate(**inp_m, max_new_tokens=512)
    gen_ids = out_m[0, inp_m["input_ids"].shape[1]:].tolist()
    arret = bool(gen_ids) and gen_ids[-1] in ids_arret_f
    ok_arret += int(arret)
    rep_m = tok_fusion.decode(gen_ids, skip_special_tokens=True).strip()
    print("=" * 72)
    print(f"[POST-MERGE V1.9] {titre_m} | {'[OK] EOS' if arret else '[ERREUR] MAX'} ({len(gen_ids)} tok)")
    print(f"Utilisateur : {q_m}\nAssistant : {rep_m}")

print("=" * 72)
print(f"Validation post-merge Kouman AI 1.9 : {ok_arret}/{len(TESTS_POST_MERGE_V19)} arrêts naturels.")